# Scraping from Lovecrafts

### Scraping every product link

In [20]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

#image_dir = "../data/images"
image_dir = "../data/images/testing"
os.makedirs(image_dir, exist_ok=True)

def scrape_product(product_link, image_dir="../data/images"):
    row = {}
    os.makedirs(image_dir, exist_ok=True)
    
    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row
    
    title_element = soup.find("h1", class_="sf-heading__title title")
    if title_element:
        raw_title = title_element.get_text(strip=True)
        row["title"] = raw_title.strip()
    else:
        row["title"] = ""

    names = soup.find_all("dt", class_="sf-property__name")
    values = soup.find_all("dd", class_="sf-property__value")

    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
        row[key] = value.get_text(strip=True)
    row["merged_info"] = " ".join(p.get_text(strip=True) for p in values)

    desc_container = soup.find("div", {"data-testid": "product-description"})
    if desc_container:
        row["description"] = desc_container.get_text(" ", strip=True)
    else:
        row["description"] = ""

    # Image Extraction and Download
    img_container = soup.find("span", class_="sf-image--wrapper sf-gallery__big-image")
    img_tag = img_container.find("img") if img_container else None
    
    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(image_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = save_path
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")
    return row


In [21]:
base_url = "https://www.lovecrafts.com/en-us/l/crochet/crochet-patterns/free-crochet-patterns?srsltid=AfmBOoryzbtxePMhSWrMgWx0Vuk-dkDGZwGOhSjdeTdpO3F9vnzuE115"
rows = []
page = 1

while True:
    print(f"Scraping page {page}...")
    url = base_url if page == 1 else f"{base_url}&p={page}"

    if page == 2:
        break

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", {"data-testid": "product-card"})
    
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link_tag = card.find("a", class_="lc-product-card__link")
        
        if link_tag:
            href = link_tag.get("href")
            row = scrape_product(href, image_dir=image_dir)
            row["product_link"] = href
            
            rows.append(row)
            time.sleep(0.5)

    page += 1


df = pd.DataFrame(rows)

df.columns = [col.replace(':', '') for col in df.columns]
print("Scraping complete!")

Scraping page 1...
Scraping page 2...
Scraping complete!


In [22]:
df.head()

,title,brand,craft,designer,format,language,notes,number_of_patterns,pages,skill_level,...,description,image_url,local_path,product_link,featured_product,crochet_hooks,finished_size,techniques_and_construction,needles_required,crochet_thread_size
0,Jellyfish Babies,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French",• Two 6 mm safety eyes,1,14,Beginner,...,"With this pattern, you will be able to make as...",https://isv.prod.lovecrafts.co/v1/images/87236...,../data/images/testing/Jellyfish_Babies.jpg,https://www.lovecrafts.com/en-us/p/jellyfish-b...,NaN,NaN,NaN,NaN,NaN,NaN
1,Sakura Market Bag,Independent Designer,Crochet,K.A.M.E. Crochet,Downloadable PDF,"Dutch, English, French, German",NaN,1,33,Intermediate,...,"NEW UPDATED VERSION CONTAINS DUTCH, FRENCH, GE...",https://isv.prod.lovecrafts.co/v1/images/cd77c...,../data/images/testing/Sakura_Market_Bag.jpg,https://www.lovecrafts.com/en-us/p/sakura-mark...,Paintbox Yarns Cotton DK,NaN,NaN,NaN,NaN,NaN
2,Easy Peasy Crochet Baby Blanket in Caron One P...,Caron,Crochet,NaN,Downloadable PDF,English,NaN,1,2,Beginner,...,Learn how to crochet with our beginners guide!,https://isv.prod.lovecrafts.co/v1/images/8a9cf...,../data/images/testing/Easy_Peasy_Crochet_Baby...,https://www.lovecrafts.com/en-us/p/easy-peasy-...,Caron One Pound,6.00mm (J),Blanket: 40in square,NaN,NaN,NaN
3,Baby Turtles,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French","• Two 6 mm safety eyes or buttons, beads, felt...",1,12,Beginner,...,"SKILLS REQUIRED Crocheting in spiral, magic ri...",https://isv.prod.lovecrafts.co/v1/images/c5cdb...,../data/images/testing/Baby_Turtles.jpg,https://www.lovecrafts.com/en-us/p/baby-turtle...,NaN,NaN,NaN,NaN,NaN,NaN
4,Squishy Baby Octopus,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French",Two 12 mm safety eyes,1,6,Beginner,...,"With this pattern, you will be able to make as...",https://isv.prod.lovecrafts.co/v1/images/95d72...,../data/images/testing/Squishy_Baby_Octopus.jpg,https://www.lovecrafts.com/en-us/p/squishy-bab...,NaN,NaN,NaN,NaN,NaN,NaN
